In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from tqdm import tqdm
from torch.nn import TransformerEncoder, TransformerEncoderLayer
import h5py
import random

# ================= 数据路径和信号参数 =================
data_path = "E:/rf_datasets/"
SNR_dB = 10
fs = 20e6
fc = 2.4e9
v = 120
apply_doppler = False
apply_awgn = False

# ================= 训练参数 =================
num_epochs = 50
patience = 5
batch_size = 256
weight_decay = 5e-4

# ================= 数据处理函数 =================
def compute_doppler_shift(v, fc):
    c = 3e8
    return (v / c) * fc

def apply_doppler_shift(signal, fd, fs):
    t = np.arange(signal.shape[-1]) / fs
    doppler_phase = np.exp(1j * 2 * np.pi * fd * t)
    return signal * doppler_phase

def add_awgn(signal, snr_db):
    signal_power = np.mean(np.abs(signal)**2)
    noise_power = signal_power / (10**(snr_db/10))
    noise = np.sqrt(noise_power/2) * (np.random.randn(*signal.shape) + 1j*np.random.randn(*signal.shape))
    return signal + noise

def load_and_preprocess(mat_folder, apply_doppler=False, target_velocity=30, apply_awgn=False, snr_db=20, fs=20e6, fc=2.4e9):
    mat_files = glob.glob(os.path.join(mat_folder, '*.mat'))
    print(f"共找到 {len(mat_files)} 个 .mat 文件")

    X_list = []
    y_list = []

    fd = compute_doppler_shift(target_velocity, fc)

    for file in tqdm(mat_files, desc='读取与处理数据'):
        with h5py.File(file, 'r') as f:
            rfDataset = f['rfDataset']
            dmrs_struct = rfDataset['dmrs'][:]
            dmrs_complex = dmrs_struct['real'] + 1j * dmrs_struct['imag']
            txID_uint16 = rfDataset['txID'][:].flatten()
            tx_id = ''.join(chr(c) for c in txID_uint16 if c != 0)

        processed_signals = []
        for sig in dmrs_complex:
            if apply_doppler:
                sig = apply_doppler_shift(sig, fd, fs)
            if apply_awgn:
                sig = add_awgn(sig, snr_db)
            iq = np.stack((sig.real, sig.imag), axis=-1)
            processed_signals.append(iq)

        processed_signals = np.array(processed_signals)
        X_list.append(processed_signals)
        y_list.append(tx_id)

    unique_labels = sorted(list(set(y_list)))
    label_to_idx = {lab: i for i, lab in enumerate(unique_labels)}
    y_idx = np.array([label_to_idx[lab] for lab in y_list])

    X_all = np.concatenate(X_list, axis=0)
    y_all = np.repeat(y_idx, dmrs_complex.shape[0])

    print(f"数据维度: X={X_all.shape}, y={y_all.shape}")
    print(f"类别映射: {label_to_idx}")
    return X_all, y_all, label_to_idx

# ================= 模型 =================
class SignalTransformer(nn.Module):
    def __init__(self, raw_input_dim, model_dim, num_heads, num_layers, num_classes, dropout=0.4):
        super(SignalTransformer, self).__init__()
        self.embedding = nn.Linear(raw_input_dim, model_dim)
        encoder_layer = TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dropout=dropout, batch_first=True)
        self.encoder = TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(model_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = self.encoder(x)
        x = x[:, -1, :]
        x = self.fc(x)
        return x

# ================= 工具函数 =================
def compute_grad_norm(model):
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item() ** 2
    return total_norm ** 0.5

def evaluate_model(model, dataloader, device, num_classes):
    model.eval()
    correct, total = 0, 0
    all_labels, all_preds = [], []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    acc = 100 * correct / total
    cm = confusion_matrix(all_labels, all_preds, labels=range(num_classes))
    return acc, cm

# ================= 随机搜索训练 =================
def random_search_train(X_all, y_all, label_to_idx, param_grid, n_iter=10):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_tensor = torch.tensor(X_all, dtype=torch.float32)
    y_tensor = torch.tensor(y_all, dtype=torch.long)
    full_dataset = TensorDataset(X_tensor, y_tensor)

    # 划分 60% 训练，20% 验证，20% 测试
    indices = np.arange(len(full_dataset))
    train_idx, temp_idx, y_train, y_temp = train_test_split(
        indices, y_all, test_size=0.4, stratify=y_all, random_state=42
    )
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5, stratify=y_temp, random_state=42
    )

    train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(Subset(full_dataset, val_idx), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(Subset(full_dataset, test_idx), batch_size=batch_size, shuffle=False)

    best_acc = 0.0
    best_params = None
    best_model_state = None

    for i in range(n_iter):
        params = {k: random.choice(v) for k, v in param_grid.items()}
        print(f"\n===== 随机搜索 {i+1}/{n_iter} 参数: {params} =====")

        model = SignalTransformer(
            raw_input_dim=2,
            model_dim=params["model_dim"],
            num_heads=params["num_heads"],
            num_layers=params["num_layers"],
            num_classes=len(label_to_idx),
            dropout=params["dropout"]
        ).to(device)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=params["learning_rate"], weight_decay=weight_decay)

        best_val_loss = float("inf")
        patience_counter = 0

        for epoch in range(num_epochs):
            model.train()
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

            # 验证
            model.eval()
            val_loss, correct_val, total_val = 0.0, 0, 0
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item()
                    _, preds = torch.max(outputs, 1)
                    total_val += labels.size(0)
                    correct_val += (preds == labels).sum().item()

            val_loss /= len(val_loader)
            val_acc = 100 * correct_val / total_val
            # print(f"Epoch {epoch+1}: Val Loss={val_loss:.4f}, Val Acc={val_acc:.2f}%")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                model_state = model.state_dict()
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print("早停触发")
                    break

        if val_acc > best_acc:
            best_acc = val_acc
            best_params = params
            best_model_state = model_state

    print(f"\n最佳验证集准确率: {best_acc:.2f}% 参数: {best_params}")

    # 用最佳参数在测试集评估
    best_model = SignalTransformer(
        raw_input_dim=2,
        model_dim=best_params["model_dim"],
        num_heads=best_params["num_heads"],
        num_layers=best_params["num_layers"],
        num_classes=len(label_to_idx),
        dropout=best_params["dropout"]
    ).to(device)
    best_model.load_state_dict(best_model_state)
    test_acc, cm = evaluate_model(best_model, test_loader, device, len(label_to_idx))
    print(f"测试集准确率: {test_acc:.2f}%")

    return best_params, best_acc, test_acc

# ================= 主函数 =================
if __name__ == "__main__":
    X_all, y_all, label_to_idx = load_and_preprocess(
        data_path,
        apply_doppler=apply_doppler,
        target_velocity=v,
        apply_awgn=apply_awgn,
        snr_db=SNR_dB,
        fs=fs,
        fc=fc
    )

    # 搜索空间
    param_grid = {
        'model_dim': [32, 64, 128, 256],
        'num_heads': [2, 4, 8],
        'num_layers': [1, 2],
        'dropout': [0.1, 0.3, 0.5],
        'batch_size': [128, 256],
        'learning_rate': [1e-4, 5e-4, 1e-3],
        'weight_decay': [1e-4, 5e-4],
        'num_epochs': [200],
        'patience': [5]
    }

    random_search_train(X_all, y_all, label_to_idx, param_grid, n_iter=30)


共找到 72 个 .mat 文件


读取与处理数据: 100%|██████████| 72/72 [00:03<00:00, 20.73it/s]


数据维度: X=(215928, 288, 2), y=(215928,)
类别映射: {'001': 0, '002': 1, '003': 2, '004': 3, '005': 4, '006': 5, '007': 6, '008': 7, '009': 8}

===== 随机搜索 1/30 参数: {'model_dim': 128, 'num_heads': 2, 'num_layers': 2, 'dropout': 0.5, 'batch_size': 128, 'learning_rate': 0.0005, 'weight_decay': 0.0001, 'num_epochs': 200, 'patience': 5} =====
Epoch 1: Val Loss=2.1957, Val Acc=12.18%
Epoch 2: Val Loss=2.1863, Val Acc=12.85%
Epoch 3: Val Loss=2.1787, Val Acc=13.51%
Epoch 4: Val Loss=2.1697, Val Acc=14.02%
Epoch 5: Val Loss=2.1689, Val Acc=14.55%
Epoch 6: Val Loss=2.1605, Val Acc=15.38%
Epoch 7: Val Loss=2.1581, Val Acc=14.41%
Epoch 8: Val Loss=2.1612, Val Acc=14.95%
Epoch 9: Val Loss=2.1558, Val Acc=14.92%
Epoch 10: Val Loss=2.1622, Val Acc=15.26%
Epoch 11: Val Loss=2.1459, Val Acc=14.29%
Epoch 12: Val Loss=2.1508, Val Acc=15.10%
Epoch 13: Val Loss=2.1414, Val Acc=15.87%
Epoch 14: Val Loss=2.1370, Val Acc=16.15%
Epoch 15: Val Loss=2.1382, Val Acc=16.43%
Epoch 16: Val Loss=2.1364, Val Acc=16.58%
Epoch